In [ ]:
# Databricks notebook source
# tutorial_name: 01-Day8-Databricks-Workflows-Jobs
# notebookName: 01-Day8-Databricks-Workflows-Jobs
# COMMAND ----------

# DBTITLE 1,Paths (same pattern as Days 5–7)
notepoint_rel = "hands-on/day-08/notebooks/01-Day8-Databricks-Workflows-Jobs.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # Change to your assigned u01–u16 (u01–u16)
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"
SOURCE_JSON = BASE_PATH + "/json/2015-summary.json"
# Same naming style as day06-{STUDENT_ID}, day07-{STUDENT_ID} — no trailing slash on root
DAY8_ROOT = f"{BASE_PATH}day08-{STUDENT_ID}"
SMOKE_OUT = f"{DAY8_ROOT}/workflow_smoke/run_proof"
AUDIT_OUT = f"{DAY8_ROOT}/workflow_smoke/job_audit_runs"
# COMMAND ----------

# DBTITLE 1,Prerequisite check (aligned with Day 6 / Day 7)
print("DAY8_ROOT:", DAY8_ROOT)
print("SMOKE_OUT:", SMOKE_OUT)
print("AUDIT_OUT:", AUDIT_OUT)
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("✓ Day 5 tables found (P_BASIC)")
except Exception as e:
    print(f"✗ Day 5 tables not found: {e}")
    print("Complete Day 5 notebook 01 first.")
try:
    spark.read.format("csv").option("header", True).load(SOURCE_CSV).limit(1).collect()
    print("✓ SOURCE_CSV readable:", SOURCE_CSV)
except Exception as e:
    print(f"✗ SOURCE_CSV: {e}")
try:
    spark.read.format("json").load(SOURCE_JSON).limit(1).collect()
    print("✓ SOURCE_JSON readable:", SOURCE_JSON)
except Exception as e:
    print("(optional) SOURCE_JSON:", type(e).__name__)
# COMMAND ----------


# Day 8 — Item 21: Databricks Workflows (Jobs)

Syllabus item 21 — jobs concepts — **tasks**, **runs**, **clusters**; hands-on **create job pipeline**.

**Sequence with earlier days:** Uses the same **`BASE_PATH`**, **`DAY5`**, **`P_BASIC`**, **`SOURCE_CSV`** variables as **Day 5** / **Day 6** / **Day 7**. Writes only under **`day08-{STUDENT_ID}/`** so you do not overwrite `day05`, `day06`, or `day07` prefixes.

**Notebook 02** covers item **22** (multi-task Bronze → Silver → Gold).


## Figure — where Jobs sit in the course

```mermaid
flowchart LR
  subgraph earlier[Days 2 to 7]
    A[PySpark code in notebooks]
    B[Delta on ABFS]
    C[DLT / Streaming optional]
  end
  A --> B
  B --> D[Day 8: Jobs / Workflows]
  D --> E[Scheduled or API-triggered runs]
  E --> B
```

Note: the smoke Delta write (Lab A) is the same Spark + Delta you already use; the **Job** only **schedules and observes** that run.


## Objectives

- Name the main objects in **Workflows / Jobs**: job, task, run, trigger, compute policy.
- Run this notebook **interactively**, then bind it to a **single-task job** and compare behavior.
- Inspect **Delta** outputs and a simple **append-only audit** log for repeated job runs.
- Know when to use an **existing cluster** vs a **job cluster** (cost vs isolation).


## Path convention (matches Days 5–7)

| Variable | Example role |
|----------|----------------|
| `BASE_PATH` | `abfss://.../data/` — shared training container |
| `DAY5` | `BASE_PATH + "/day5"` — earlier labs |
| `P_BASIC` | Delta table from Day 5 |
| `DAY8_ROOT` | `BASE_PATH + "day08-" + STUDENT_ID` — **your** sandbox for today |

Outputs today: `.../day08-u25/workflow_smoke/...` (adjust `STUDENT_ID`).


## Why use a Job instead of only running a notebook?

| Interactive notebook | Job |
|----------------------|-----|
| You click **Run** | Scheduler or API triggers **Run now** |
| Stops when you disconnect | Intended for **production** and **SLAs** |
| Harder to audit centrally | **Runs** tab: history, duration, retries |
| Same code cells | Can pass **parameters**, use **task libraries** |

Jobs are how you **operationalize** the same PySpark you wrote on Days 2–7.


## Figure — interactive session vs job run

```mermaid
flowchart TB
  subgraph inter[Interactive notebook]
    U1[You] --> NB[Run cells]
    NB --> C1[Cluster]
    C1 --> ABFS[(ABFS paths)]
  end
  subgraph job[Job run]
    U2[Schedule or Run now] --> JR[Job run]
    JR --> T[Task: notebook]
    T --> C2[Cluster]
    C2 --> ABFS2[(Same ABFS paths)]
  end
```

Same code paths (`BASE_PATH`, `day08-...`); different **entry point** and **run history**.


## Glossary (UI-aligned)

- **Job** — Named container for tasks + optional schedules and notifications.
- **Task** — One step (e.g. **Notebook**, **Python file**, **DLT pipeline**, **SQL**, **dbt**).
- **Run** — Single execution attempt; contains per-task status and logs.
- **Trigger** — Schedule (cron), **file arrival**, **continuous** (for some task types), or manual.
- **Compute** — Cluster or SQL warehouse attached to the task; can be **existing** or **job**-scoped.


## Common task types (high level)

1. **Notebook** — What you use in this course most often.
2. **Delta Live Tables** — Declarative pipelines (Day 7).
3. **SQL** — Warehouse query or file.
4. **Python / wheel** — Packaged tasks.

Multi-task graphs are item **22** (see notebook `02`).


## Figure — Job, tasks, and a single run

```mermaid
flowchart TB
  J[Job: day08_smoke]
  J --> T1[Task: notebook 01]
  J --> TR[Triggers optional]
  R[Run #12345] --> T1
  T1 --> L[Logs + result state]
```

**ASCII (compact):**

```text
  Job
   └── Task (notebook path + cluster)
         └── each Run: Pending → Running → Succeeded/Failed
```


In [ ]:
# Runtime context (useful when comparing interactive vs job runs)
import pyspark
print("Spark version:", spark.version)
print("Python:", pyspark.__version__)
try:
    cid = spark.conf.get("spark.databricks.clusterUsageTags.clusterId", "(not set)")
    print("ClusterId tag:", cid)
except Exception:
    pass


## Run lifecycle (what you see on the Runs page)

Typical states: **Pending** → **Running** → **Succeeded** or **Failed** / **Timeout** / **Cancelled**.  
If you enable **retries**, a failed task may transition to **Retrying** before final failure.

After you create a job, open **Run**, expand each **task**, and download **logs** when you need to debug a failure.


## Figure — run state machine (concept)

```mermaid
stateDiagram-v2
  [*] --> Pending
  Pending --> Running
  Running --> Succeeded
  Running --> Failed
  Running --> Cancelled
  Failed --> Retrying : if retries left
  Retrying --> Running
  Succeeded --> [*]
  Failed --> [*]
  Cancelled --> [*]
```


## Lab A — Write a deterministic “smoke” Delta table

This cell is the main **body** your **first job task** should execute. It **overwrites** `SMOKE_OUT` each time so reruns stay predictable.


In [ ]:
import uuid

JOB_CORRELATION = str(uuid.uuid4())[:12]
print("JOB_CORRELATION:", JOB_CORRELATION)

spark.sql(
    f"SELECT '{STUDENT_ID}' AS student_id, current_timestamp() AS ran_at, "
    f"'{JOB_CORRELATION}' AS correlation_id"
).write.format("delta").mode("overwrite").save(SMOKE_OUT)

spark.read.format("delta").load(SMOKE_OUT).show(truncate=False)


In [ ]:
# Exercise (concept check): build the SQL string the smoke cell will run — no write yet
demo_sql = (
    f"SELECT '{STUDENT_ID}' AS student_id, current_timestamp() AS ran_at, 'dry-run' AS correlation_id"
)
print(demo_sql)
spark.sql(demo_sql).show(truncate=False)


The cell above ran a **read-only** preview of the same shape as the smoke table. The next cell **materializes** it to Delta at `SMOKE_OUT`.


## Lab B — Delta history on the smoke table

Jobs should produce **observable** side effects. Use **`DESCRIBE HISTORY`** to see each job run as a new table version (if the write committed).


In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{SMOKE_OUT}`").select(
    "version", "operation", "operationMetrics"
).show(15, truncate=80)


## Figure — each job run adds Delta history (concept)

```text
  Version 0: first WRITE (overwrite)
  Version 1: second job run → another WRITE
  ...
  Audit table: APPEND rows (one per run) → grows
```


## Lab C — Append-only audit log (multiple job runs)

Unlike `SMOKE_OUT` (overwrite), **`AUDIT_OUT`** uses **`append`** so repeated job runs accumulate rows.  
Use this pattern when you need a lightweight **run registry** without a separate database.


In [ ]:
import uuid
_corr = globals().get("JOB_CORRELATION") or str(uuid.uuid4())[:12]
audit_row = spark.sql(
    f"SELECT '{STUDENT_ID}' AS student_id, current_timestamp() AS logged_at, "
    f"'{_corr}' AS correlation_id, 'smoke_task' AS task_name"
)
audit_row.write.format("delta").mode("append").save(AUDIT_OUT)

print("Audit tail:")
spark.read.format("delta").load(AUDIT_OUT).orderBy("logged_at", ascending=False).show(10, truncate=False)


## Lab D — Table metadata (`DESCRIBE DETAIL`)

Useful in class: show **format**, **partitionColumns**, **numFiles** — same Delta vocabulary as **Day 5**.


In [ ]:
spark.sql(f"DESCRIBE DETAIL delta.`{SMOKE_OUT}`").show(truncate=False)


In [ ]:
# Exercise: compare row counts smoke vs audit (after Labs A–C)
try:
    s = spark.read.format('delta').load(SMOKE_OUT).count()
    a = spark.read.format('delta').load(AUDIT_OUT).count()
    print(f'smoke rows (latest overwrite): {s}')
    print(f'audit rows (cumulative appends): {a}')
except Exception as e:
    print('Run Labs A–C first:', e)


## Parameterizing jobs

In the job UI, **Task parameters** map to **`dbutils.widgets`** in the notebook (notebook **02** uses `pipeline_stage`).  
For this notebook you can add optional widgets later (e.g. `dry_run`) — pattern:

```python
dbutils.widgets.text("dry_run", "false")
```

Then read with `dbutils.widgets.get("dry_run")` and branch in code.


## Checklist — **single-task job** (item 21)

1. Run this notebook **interactively** top-to-bottom once; confirm no errors.
2. **Workflows** → **Jobs** → **Create job**.
3. Name: `day08_smoke_{STUDENT_ID}` (or your standard).
4. **Add task** → **Notebook** → pick **this notebook**.
5. **Compute:** existing cluster (class) or job cluster (production pattern).
6. **Run now** → wait for **Succeeded**.
7. Re-run the job → check **`AUDIT_OUT`** gained a new row; **`SMOKE_OUT`** still has one logical row (overwrite).
8. Optional: **Job settings** → **Notifications** on failure.
9. Optional: duplicate task (sequential) to simulate a two-step job before moving to **02**.


## Existing cluster vs job cluster

| | Existing / shared cluster | Job cluster |
|---|---------------------------|-------------|
| **Pros** | Fast start in class; shared cost | Isolated; autoscale; terminates when idle |
| **Cons** | Queuing; shared state | Cold start; need policy permissions |

For **production**, prefer **job clusters** created from a **policy** your admin defines.


## Permissions (typical)

- **CAN MANAGE RUN** on the job (or owner).
- **Attach** policy / cluster if using job compute.
- **READ/WRITE** on `abfss` paths (your app registration / credential chain from **`02-Mount-Azure-Data-Lake`** on other days).

If the job fails with **forbidden** on storage, fix **OAuth / service principal** the same way as Day 5 interactive runs.


## Troubleshooting

| Symptom | Things to check |
|---------|-----------------|
| Task **Pending** forever | Cluster not running; **max workers**; policy cap |
| **ImportError** in job | Cluster **libraries** differ from your notebook session |
| **Path not found** | `STUDENT_ID` typo; wrong `BASE_PATH`; storage ACL |
| **Delta** errors | Concurrent overwrite; run **VACUUM** only when taught (Day 6) |

**Extra reading:** `databricks/**/13-Building-n-Managing-Workflows-with-Databricks*`, `**/Job_Scheduling_and_Monitoring*`.


## Next step

Open **`02-Day8-Medallion-MultiTask-Workflow.ipynb`** for item **22**: **three** tasks, **dependencies**, and a full **Bronze → Silver → Gold** flow using the same **`BASE_PATH`** / **`STUDENT_ID`** pattern.


## Optional — schedule (concept)

Cron examples (UTC, illustrative):

- Daily 06:00: `0 0 6 * * ?`
- Every 6 hours: `0 0 */6 * * ?`

Configure in the job’s **Triggers / Schedule**; **pause** schedules in shared training workspaces when not teaching.


In [ ]:
# Optional cleanup (your prefix only) — uncomment in your workspace when needed
# dbutils.fs.rm(DAY8_ROOT, recurse=True)
